In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df_type_repairs = pd.read_csv("../../datasets/type_repairs.csv")

df_type_repairs

In [ ]:
len(df_type_repairs[(df_type_repairs['Constraint Deleted'] == True)] )

In [ ]:
len(df_type_repairs[(df_type_repairs['Constraint Deprecated'] == True)] )

In [ ]:
len(df_type_repairs[(df_type_repairs['Included as Exception'] == True)] )

In [ ]:
df_type_repairs['A-box wdt statement Deleted'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementDeleted(row):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{row['subject']}> <{wdt_pid}> [] }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return 88

# Example usage
print(statementDeleted(df_type_repairs.iloc[0]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_type_repairs.iterrows(), total=len(df_type_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['A-box wdt statement Deleted']):
        result = statementDeleted(row)
        if result == 88:
            print("index value:")
            print(index)
        else:
            df_type_repairs.at[index, 'A-box wdt statement Deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 1000000 == 0 and index != 0:
        df_type_repairs.to_csv("checkpoint_type.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_type_repairs.to_csv("checkpoint_type.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_type_repairs.iloc[4919]

In [ ]:
df_type_repairs = df_type_repairs.drop(4919)

In [ ]:
len(df_type_repairs[(df_type_repairs['A-box wdt statement Deleted'] == True)] )

In [ ]:
df_type_repairs[
    (df_type_repairs['Constraint Deleted'] == False)
    & (df_type_repairs['Constraint Deprecated'] == False)
    & (df_type_repairs['Included as Exception'] == False)
    & (df_type_repairs['A-box wdt statement Deleted'] == False)
]

In [ ]:
df_type_repairs['constraint_type'].value_counts()

In [ ]:
df_type_repairs.iloc[3]

In [ ]:
getType(df_type_repairs.iloc[3], "ENTER_qEndpoint_WD_2019")

In [ ]:
getType(df_type_repairs.iloc[3], "ENTER_qEndpoint_WD_2023")

In [ ]:
df_type_repairs['t-box hierarchy added'] = None
df_type_repairs['a-box type statement added'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getType(row, endpoint):

    if row['constraint_type'] == 'instance_of':
        string_path_type = '<'+ row['subject']+'> wdt:P31 ?type.'
    elif row['constraint_type'] == 'subclass_of':
        string_path_type = '<'+ row['subject']+'> wdt:P279 ?type.'
    elif row['constraint_type'] == 'instance_or_subclass':
        string_path_type = '<'+ row['subject']+'> wdt:P31|wdt:P279 ?type.'
    else:
        return None

    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT 
            ?type
        {{
          {string_path_type}
        }}
            """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
       # Parse the XML
        root = ET.fromstring(response.text)

        # Define the namespace
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Create an empty set to store the type URIs
        type_set = set()

        # Find all URI elements corresponding to the variable 'type'
        for uri in root.findall(".//ns:binding[@name='type']/ns:uri", namespace) or []:
            type_set.add(uri.text)
            
        return type_set
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return set()
    
def typeStatementAdded(row):
    setType19 = getType(row, "ENTER_qEndpoint_WD_2019")
    setType23 = getType(row, "ENTER_qEndpoint_WD_2023")
    if setType19 is None or setType23 is None:
        return None
    if len(setType23) == 0:
        return False
    if setType19 == setType23:
        return False
    if len(setType23) > len(setType19):
        return True
    if setType19 != setType23:
        return True
    
    
def hierarchyAdded(row):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"
    
    # get the types in 2019
    set_type = getType(row, "ENTER_qEndpoint_WD_2019")
    if len(set_type) == 0:
        return False

    for class_type in set_type:
        
        # regardless of the constraint type, they all are now checked with subclass path because the first step was calculated
        # in the getType function
        string_path_type = '<'+ class_type +'> wdt:P279* ?class.'
    
        # SPARQL query
        query = f"""
            PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX wikibase: <http://wikiba.se/ontology#>
            PREFIX p: <http://www.wikidata.org/prop/>
            PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
            PREFIX ps: <http://www.wikidata.org/prop/statement/>
            PREFIX wd: <http://www.wikidata.org/entity/>
            PREFIX wdt: <http://www.wikidata.org/prop/direct/>

            ASK
            {{
              <{row['property']}> p:P2302 ?statement.
              ?statement ps:P2302 wd:Q21503250.
              ?statement pq:P2308 ?class.
              {string_path_type}
            }}
        """
        #print(query)
        # URL encode the query
        encoded_query = requests.utils.quote(query)

        # Build the complete URL
        url = f"{endpoint}?query={encoded_query}"

        # Send HTTP GET request
        headers = {"Accept": "application/xhtml+xml,application/xml;"}

        response = requests.get(url,headers=headers)
        #print(response.text)
        # Check if the request was successful and parse the response
        if response.ok:
            # Parse the XML response
            #print(response.text)
            root = ET.fromstring(response.text)
            boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
            if boolean_element is not None:
                #print(boolean_element.text)
                if boolean_element.text.lower() == 'true':
                    return True
            else:
                print("Error: 'boolean' element not found in XML response")
        else:
            # If there's an error in the request, return None
            print("Error:", response)
            print(row)
            
    return False

# Example usage
#print(hierarchyAdded(df_type_repairs.iloc[2]))
#print(getType(df_type_repairs.iloc[2], "ENTER_qEndpoint_WD_2019"))
print(typeStatementAdded(df_type_repairs.iloc[3]))
print(hierarchyAdded(df_type_repairs.iloc[3]))

In [ ]:
set1 = {1,2,3}
set2 = {2,3,1}
set1 == set2

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_type_repairs.iterrows(), total=len(df_type_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['t-box hierarchy added']):
        result = hierarchyAdded(row)
        df_type_repairs.at[index, 't-box hierarchy added'] = result
        
    if pd.isna(row['a-box type statement added']):
        result = typeStatementAdded(row)
        df_type_repairs.at[index, 'a-box type statement added'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_type_repairs.to_csv("checkpoint_type.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_type_repairs.to_csv("checkpoint_type.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_type_repairs['t-box hierarchy added'].value_counts()

In [ ]:
df_type_repairs[
    (df_type_repairs['Constraint Deleted'] == False)
    & (df_type_repairs['Constraint Deprecated'] == False)
    & (df_type_repairs['Included as Exception'] == False)
    & (df_type_repairs['A-box wdt statement Deleted'] == False)
    & (df_type_repairs['t-box hierarchy added'] == False)
]

In [ ]:
df_type_repairs['t-box hierarchy added'].value_counts()

In [ ]:
df_type_repairs['a-box type statement added'].value_counts()

In [ ]:
df_type_repairs

In [ ]:
df_type_repairs[
    (df_type_repairs['Constraint Deleted'] == False)
    & (df_type_repairs['Constraint Deprecated'] == False)
    & (df_type_repairs['Included as Exception'] == False)
    & (df_type_repairs['A-box wdt statement Deleted'] == False)
    & (df_type_repairs['t-box hierarchy added'] == False)
    & (df_type_repairs['a-box type statement added'] == False)
]

In [ ]:
df_type_repairs['t-box constraint type changed'] = None

In [ ]:
df_type_repairs

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getRelation(row, endpoint):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    
    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT ?relation
        {{
          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21503250. ## subject-type constraint
          ?statement pq:P2309 ?relation.
          # no deprecated, no exceptions
          FILTER NOT EXISTS {{ ?statement pq:P2241 [] }}
          FILTER NOT EXISTS {{ ?statement wikibase:rank wikibase:DeprecatedRank }}
        }}
            """
    #print(query)
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element =  root.find(".//ns:binding[@name='relation']/ns:uri", namespace)
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return 88
    
def hasRelationChanged(row):
    relation_2019 = getRelation(row, 'ENTER_qEndpoint_WD_2019')
    if relation_2019 is None:
        return None
    relation_2023 = getRelation(row, 'ENTER_qEndpoint_WD_2023')
    if relation_2023 is None:
        return None
    if relation_2019 != relation_2023:
        return True
    return False

# Example usage
print(getRelation(df_type_repairs.iloc[4], 'ENTER_qEndpoint_WD_2019'))
print(getRelation(df_type_repairs.iloc[4], 'ENTER_qEndpoint_WD_2023'))
print(hasRelationChanged(df_type_repairs.iloc[4]))

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_type_repairs.iterrows(), total=len(df_type_repairs)):
    
    if bool(row['Constraint Deleted']) is True:
        df_type_repairs.at[index, 't-box constraint type changed'] = False
    elif pd.isna(row['t-box constraint type changed']):
        result = hasRelationChanged(row)
        if result == 88:
            print("index value:")
            print(index)
        else:
            df_type_repairs.at[index, 't-box constraint type changed'] = result

    # Save a checkpoint every 10,000 rows
    if index % 300000 == 0 and index != 0:
        df_type_repairs.to_csv("checkpoint_type_2.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_type_repairs.to_csv("checkpoint_type_2.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
df_type_repairs = pd.read_csv("checkpoint_type_2.csv")

df_type_repairs

In [ ]:
df_type_repairs['t-box constraint type changed'].value_counts()

In [ ]:
df_type_repairs['t-box constraint type changed'].value_counts()

In [ ]:
df_type_repairs['t-box constraint type changed'].value_counts()

In [ ]:
df_unknown = df_type_repairs[
    (df_type_repairs['Constraint Deleted'] == False)
    & (df_type_repairs['Constraint Deprecated'] == False)
    & (df_type_repairs['Included as Exception'] == False)
    & (df_type_repairs['A-box wdt statement Deleted'] == False)
    & (df_type_repairs['t-box hierarchy added'] == False)
    & (df_type_repairs['a-box type statement added'] == False)
    & (df_type_repairs['t-box constraint type changed'] == False)
]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isStillViolation(row, endpoint):
    
    wdt_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # URL of the endpoint
    
    if row['constraint_type'] == 'instance_of':
        string_path_type = '<'+ row['subject']+'> wdt:P31/wdt:P279* ?allowed_type.'
    elif row['constraint_type'] == 'subclass_of':
        string_path_type = '<'+ row['subject']+'> wdt:P279+ ?allowed_type.'
    elif row['constraint_type'] == 'instance_or_subclass':
        string_path_type = '<'+ row['subject']+'> wdt:P31/wdt:P279*|wdt:P279+ ?allowed_type.'
    else:
        return None
    
    c_type = ''
    if bool(row['t-box constraint type changed']) == False:
        if row['constraint_type'] == 'instance_of':
            c_type = '?statement pq:P2309 wd:Q21503252.'
        elif row['constraint_type'] == 'subclass_of':
            c_type = '?statement pq:P2309 wd:Q21514624.'
        else:
            c_type = '?statement pq:P2309 wd:Q30208840.'
    
    # SPARQL query
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        ASK
        {{
          ?subject <{wdt_pid}> [].

          <{row['property']}> p:P2302 ?statement.
          ?statement ps:P2302 wd:Q21503250. ## type constraint
          {c_type}

          # no deprecated, no exceptions
          FILTER NOT EXISTS {{?statement pq:P2241 []}}
          FILTER NOT EXISTS {{?statement wikibase:rank wikibase:DeprecatedRank}}
          FILTER NOT EXISTS {{?statement pq:P2303 ?subject}}
          FILTER (?subject = <{row['subject']}>)
          FILTER NOT EXISTS {{
            ?statement pq:P2308 ?allowed_type.
            {string_path_type}
          }}
        }}
            """
    #print(query)
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return 88

# Example usage

In [ ]:
isStillViolation(df_unknown.iloc[2], "ENTER_qEndpoint_WD_2023")

In [ ]:
df_unknown.iloc[0]

In [ ]:
df_unknown['still_violation'] = None

In [ ]:
df_unknown

In [ ]:
df_type_repairs

In [ ]:
df_type_repairs.to_csv("all_repairs_1.csv", index=False)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Process the rows with progress bar
for index, row in tqdm(df_unknown.iterrows(), total=len(df_unknown)):
    
    # Check for unprocessed rows
    if pd.isna(row['still_violation']) or row['still_violation'] == None:
        result = isStillViolation(row, "ENTER_qEndpoint_WD_2023")
        if result == 88:
            print("index value:")
            print(index)
        else:
            df_unknown.at[index, 'still_violation'] = result

    # Save a checkpoint every 10,000 rows
    if index % 100000 == 0 and index != 0:
        df_unknown.to_csv("checkpoint_unknown.csv", index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_unknown.to_csv("checkpoint_unknown.csv", index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_unknown['still_violation'].value_counts()

In [ ]:
# Step 1: Get indices of rows in df_unknown where still_violation is True
violation_indices = df_unknown[df_unknown['still_violation'] == True].index

violation_indices

In [ ]:
df_unknown

In [ ]:
# Step 2: Remove rows with these indices from df_type_repairs
df_type_repairs = df_type_repairs.drop(violation_indices)

In [ ]:
df_false = df_unknown[
    (df_unknown['still_violation'] == False)
]

In [ ]:
df_false.iloc[0]

In [ ]:
typeStatementAdded(df_false.iloc[0])

In [ ]:
setType19 = getType(df_false.iloc[0], "ENTER_qEndpoint_WD_2019")
setType23 = getType(df_false.iloc[0], "ENTER_qEndpoint_WD_2023")

print(setType19)
print(setType23)

In [ ]:
df_type_repairs

In [ ]:
df_type_repairs['t-box type added to constraint'] = None

In [ ]:
df_type_repairs['t-box types added to constraint'] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getConstraintExpectedTypes(row, endpoint):

    if row['constraint_type'] == 'instance_of':
        string_path_type = '<'+ row['subject']+'> wdt:P31 ?type.'
    elif row['constraint_type'] == 'subclass_of':
        string_path_type = '<'+ row['subject']+'> wdt:P279 ?type.'
    elif row['constraint_type'] == 'instance_or_subclass':
        string_path_type = '<'+ row['subject']+'> wdt:P31|wdt:P279 ?type.'
    else:
        return None

    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT 
            ?type
        {{
          {string_path_type}
        }}
            """
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
       # Parse the XML
        root = ET.fromstring(response.text)

        # Define the namespace
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Create an empty set to store the type URIs
        type_set = set()

        # Find all URI elements corresponding to the variable 'type'
        for uri in root.findall(".//ns:binding[@name='type']/ns:uri", namespace) or []:
            type_set.add(uri.text)
            
        return type_set
    else:
        # If there's an error in the request, return None
        print("Error:", response)
        print(row)
        return set()